In [5]:
import pandas as pd
import numpy as np

# Chargement du dataset clean
df = pd.read_csv(r"C:\Users\ouard\Smart_Energy_AI\03_Consumption_Prediction\data\household_power_consumption_cleaned.csv")

# Conversion du timestamp
df["Datetime"] = pd.to_datetime(df["Datetime"])

# Vérification
print("Dimensions :", df.shape)
print("Période :", df["Datetime"].min(), "→", df["Datetime"].max())

Dimensions : (2075259, 12)
Période : 2006-12-16 17:24:00 → 2010-11-26 21:02:00


In [6]:
# Définition des limites temporelles
end_date = df["Datetime"].max()

test_start = end_date - pd.DateOffset(years=1)
validation_start = test_start - pd.DateOffset(months=6)

# Séparation chronologique
train_df = df[df["Datetime"] < validation_start]

validation_df = df[
    (df["Datetime"] >= validation_start) &
    (df["Datetime"] < test_start)
]

test_df = df[df["Datetime"] >= test_start]

# Affichage des périodes et dimensions
for name, data in [
    ("TRAIN", train_df),
    ("VALIDATION", validation_df),
    ("TEST", test_df)
]:
    print(f"\n--- {name} ---")
    print("Début :", data["Datetime"].min())
    print("Fin   :", data["Datetime"].max())
    print("Dimensions :", data.shape)


--- TRAIN ---
Début : 2006-12-16 17:24:00
Fin   : 2009-05-26 21:01:00
Dimensions : (1284698, 12)

--- VALIDATION ---
Début : 2009-05-26 21:02:00
Fin   : 2009-11-26 21:01:00
Dimensions : (264960, 12)

--- TEST ---
Début : 2009-11-26 21:02:00
Fin   : 2010-11-26 21:02:00
Dimensions : (525601, 12)


In [7]:
# Sélection des variables utilisées pour la prédiction
target = "Global_active_power"

print("Variable cible :", target)
print("Train :", train_df.shape)
print("Validation :", validation_df.shape)
print("Test :", test_df.shape)

Variable cible : Global_active_power
Train : (1284698, 12)
Validation : (264960, 12)
Test : (525601, 12)


In [8]:
# Définition des variables

features = [
    "Global_reactive_power",
    "Voltage",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

target = "Global_active_power"

X_train = train_df[features]
y_train = train_df[target]

X_val = validation_df[features]
y_val = validation_df[target]

X_test = test_df[features]
y_test = test_df[target]

print("X_train :", X_train.shape, "| y_train :", y_train.shape)
print("X_val   :", X_val.shape, "| y_val   :", y_val.shape)
print("X_test  :", X_test.shape, "| y_test  :", y_test.shape)

X_train : (1284698, 6) | y_train : (1284698,)
X_val   : (264960, 6) | y_val   : (264960,)
X_test  : (525601, 6) | y_test  : (525601,)


In [11]:
# Export Excel avec formatage et découpage automatique

def export_excel(df, filename, sheet_prefix):
    max_rows = 1_048_575  # lignes de données, en réservant 1 ligne aux en-têtes

    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        for i in range(0, len(df), max_rows):
            chunk = df.iloc[i:i + max_rows]

            sheet_name = f"{sheet_prefix}_{i // max_rows + 1}"
            chunk.to_excel(writer, index=False, sheet_name=sheet_name)

            ws = writer.sheets[sheet_name]
            ws.freeze_panes = "A2"
            ws.auto_filter.ref = ws.dimensions

            # Ajustement simple de la largeur des colonnes
            for column in ws.columns:
                column_letter = column[0].column_letter
                header_length = len(str(column[0].value))
                ws.column_dimensions[column_letter].width = max(header_length + 3, 15)

# Export des trois ensembles
export_excel(train_df, "train.xlsx", "Training")
export_excel(validation_df, "validation.xlsx", "Validation")
export_excel(test_df, "test.xlsx", "Test")

print("Export Excel terminé.")

Export Excel terminé.


In [14]:
# Rechargement des datasets sauvegardés

train_df = pd.read_excel("train.xlsx")
validation_df = pd.read_excel("validation.xlsx")
test_df = pd.read_excel("test.xlsx")

# Conversion de la colonne temporelle
for dataset in [train_df, validation_df, test_df]:
    dataset["Datetime"] = pd.to_datetime(dataset["Datetime"])

print("Datasets rechargés avec succès.")
print("Train :", train_df.shape)
print("Validation :", validation_df.shape)
print("Test :", test_df.shape)

Datasets rechargés avec succès.
Train : (1048575, 12)
Validation : (264960, 12)
Test : (525601, 12)
